In [15]:
from langgraph.graph import StateGraph,START,END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict,Literal
from dotenv import load_dotenv
load_dotenv()
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
)

In [16]:
class EquationState(TypedDict):
    a:int
    b:int
    c:int
    equation:str
    discriminant:float
    result:str

In [23]:
def show_equation(state:EquationState)->EquationState:
    equation =f"{state["a"]}x^2+{state["b"]}x+{state["c"]}=0"
    return {
        "equation":equation
    }

def calculate_discriminant(state:EquationState)->EquationState:
    discriminant = state["b"]**2-4*state["a"]*state["c"]
    return {
        "discriminant":discriminant
    }

def real_roots(state:EquationState)->EquationState:
    root1 =(-state["b"]+state["discriminant"]**0.5)/(2*state["a"])
    root2=(-state["b"]-state["discriminant"]**0.5)/(2*state["a"])
    return {
        "result":f"The roots are {root1} and {root2}"
    }

def repeated_roots(state:EquationState)->EquationState:
    root = -state["b"]/(2*state["a"])
    return {
        "result":f"The roots are {root}"
    }

def no_real_roots(state:EquationState)->EquationState:
    return {
        "result":"The roots are imaginary"
    }

def check_condition(state: EquationState) -> Literal[
    "real_roots",
    "repeated_roots",
    "no_real_roots"
]:
    if state["discriminant"] > 0:
        return "real_roots"

    elif state["discriminant"] == 0:
        return "repeated_roots"

    else:
        return "no_real_roots"

In [24]:
graph = StateGraph(EquationState)

graph.add_node("show_equation",show_equation)
graph.add_node("calculate_discriminant",calculate_discriminant)
graph.add_node("real_roots",real_roots)
graph.add_node("repeated_roots",repeated_roots)
graph.add_node("no_real_roots",no_real_roots)

graph.add_edge(START,"show_equation")
graph.add_edge("show_equation","calculate_discriminant")
graph.add_conditional_edges("calculate_discriminant",check_condition)
graph.add_edge("real_roots",END)
graph.add_edge("repeated_roots",END)
graph.add_edge("no_real_roots",END)


workflow =graph.compile()

In [25]:
initial_state = {
    "a":4,
    "b":-5,
    "c":-5,
}

final_state = workflow.invoke(initial_state)
print(final_state)

{'a': 4, 'b': -5, 'c': -5, 'equation': '4x^2+-5x+-5=0', 'discriminant': 105, 'result': 'The roots are 1.9058688457449497 and -0.6558688457449497'}
